## Setting Up

In [1]:
# IMPORTS
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipyvolume as ipv
import neuropythy as ny

import torch
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.utils import to_undirected
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SplineConv

libpath = '/home/achegu/repos/mesh-annot/src'
if libpath not in sys.path:
    sys.path.append(libpath)

import mesh_annot

/home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_scatter/_version_cpu.so
  import torch_geometric.typing
/home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch_sparse/_version_cpu.so
  import torch_geometric.typing


In [2]:
# CONFIGURE FREESURFER HOME
ny.config['freesurfer_home'] = '/opt/freesurfer'
ds = ny.config['freesurfer_home']

In [3]:
%matplotlib inline

## Basic Neuropythy Stuff

In [4]:
subject = ny.freesurfer_subject('fsaverage3')
fsaverage = ny.freesurfer_subject('fsaverage')

In [5]:
# SPLITTING INTO HEMISPHERES
subject_lh = subject.hemis['lh']
subject_rh = subject.hemis['rh']

fsaverage_lh = fsaverage.hemis['lh']
fsaverage_rh = fsaverage.hemis['rh']

In [6]:
# ASSIGNING SURFACE INFORMATION
lh_surface = subject_lh.surfaces['white']
rh_surface = subject_rh.surfaces['white']

In [7]:
# PLOT THE SURFACE
fig = ipv.figure(width=900, height=300)
ny.cortex_plot(lh_surface, color='thickness', cmap='cool', figure=fig)
ny.cortex_plot(rh_surface, color='thickness', cmap='cool', figure=fig)
ipv.show()

Container(figure=Figure(box_center=[0.5, 0.5, 0.5], box_size=[1.0, 1.0, 1.0], camera=PerspectiveCamera(fov=0.6…

In [8]:
# VERTICES OF THE HEMISPHERES
lh_vertices = lh_surface.coordinates
rh_vertices = rh_surface.coordinates

In [9]:
# MESHES OF THE HEMISPHERES
lh_faces = lh_surface.tess.faces
rh_faces = rh_surface.tess.faces

In [10]:
# EDGES OF THE HEMISPHERES
rh_edges = lh_surface.tess.edges
lh_edges = rh_surface.tess.edges

## Loading In Data

In [ ]:
# TODO When loading in multiple subjects, interpolate that information onto the fsaverage maps

In [11]:
# TODO Find a way to load both hemispheres
    # Currently, I'm thinking of loading in both hemisphere's edges and 
    # then transposing/reordering the right hemisphere's edges such that 
    # all of them come after the left hemisphere. Then, I'll probably 
    # have to add another column with a mask (or something similar) to 
    # differentiate between the left and right hemispheres.
if lh_edges.shape[0] == 2 and lh_edges.shape[1] != 2:
    lh_edges = lh_edges.T

if rh_edges.shape[0] == 2 and rh_edges.shape[1] != 2:
    rh_edges = rh_edges.T

# Build the feature object
edge_index = torch.tensor(rh_edges.T, dtype=torch.long)
edge_index = to_undirected(edge_index) # Creating bidirectional edges
x_coords = torch.tensor(rh_vertices.T, dtype=torch.float)

# --- Needed some help with this ---
src = x_coords[edge_index[0]]
trg = x_coords[edge_index[1]]
edge_attr = src - trg
edge_attr = (edge_attr - edge_attr.min())/(edge_attr.max() - edge_attr.min()) # Normalize edge_attr
# -----------------------------------

graph_data = Data(x=x_coords, edge_index=edge_index, edge_attr=edge_attr)
# TODO Check if graph_data is a NoneType here
# print(type(graph_data))

# Build the target
target_thickness = rh_surface.properties['thickness']
# TODO Is there an easier way to do this?
target_thickness = target_thickness.byteswap().view(target_thickness.dtype.newbyteorder())
graph_data.y = torch.tensor(target_thickness, dtype=torch.float).unsqueeze(1)

print(graph_data)

Data(x=[642, 3], edge_index=[2, 3840], edge_attr=[3840, 3], y=[642, 1])


In [ ]:
# TODO Create dataset

In [ ]:
# TODO Split dataset into training, validation, and test

## Building Model

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# TODO SplineGCN 
model = mesh_annot.SplineGCN(in_channels=3, out_channels=1).to(device)
graph_data = graph_data.to(device)

lr = 0.01
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

model.train()

for epoch in range(200):
    optimizer.zero_grad()
    output = model(graph_data)
    loss = criterion(output, graph_data.y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} || MSE Loss: {loss.item():.4f}')

/home/achegu/.conda/envs/mesh-annot-env/lib/python3.12/site-packages/torch/nn/modules/module.py:1789: UserWarning: We do not recommend using the non-optimized CPU version of `SplineConv`. If possible, please move your data to GPU.
  return forward_call(*args, **kwargs)


Epoch 000 || MSE Loss: 3.1871
Epoch 010 || MSE Loss: 0.3606
Epoch 020 || MSE Loss: 0.3011
Epoch 030 || MSE Loss: 0.2480
Epoch 040 || MSE Loss: 0.2123
Epoch 050 || MSE Loss: 0.1767
Epoch 060 || MSE Loss: 0.1440
Epoch 070 || MSE Loss: 0.1149
Epoch 080 || MSE Loss: 0.0907
Epoch 090 || MSE Loss: 0.0693
Epoch 100 || MSE Loss: 0.0523
Epoch 110 || MSE Loss: 0.0403
Epoch 120 || MSE Loss: 0.0312
Epoch 130 || MSE Loss: 0.0246
Epoch 140 || MSE Loss: 0.0197
Epoch 150 || MSE Loss: 0.0161
Epoch 160 || MSE Loss: 0.0133
Epoch 170 || MSE Loss: 0.0111
Epoch 180 || MSE Loss: 0.0094
Epoch 190 || MSE Loss: 0.0080


## Analysing Results

In [ ]:
# SANITY CHECKS
# print("GCN output shape:", output.shape)
# print("Output type:", type(output))
# print("Contains NaNs?:", torch.isnan(output).any().item())
# print("Min value:", output.min().item())
# print("Max value:", output.max().item())
# print("Mean value:", output.mean().item())

In [14]:
model.eval()
with torch.no_grad():
    # TODO Something is wrong here, I have a hunch that it's just printing the regular graph_data.
    predictions = model(graph_data).cpu().numpy().flatten()

fig = ipv.figure(width=900, height=300)
ny.cortex_plot(rh_surface, color=predictions, cmap='cool')

Figure(box_center=[0.5, 0.5, 0.5], box_size=[1.0, 1.0, 1.0], camera=PerspectiveCamera(fov=0.644570721372708, p…